In [1]:
#STEP 1:IMPORT LIBRARIES
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression


In [2]:
#STEP 2:LOAD DATASET
data = pd.read_csv("Dataset/covid.csv")
data.shape


(566602, 23)

In [3]:
#STEP 3:CREATE DEATH LABEL
data['death'] = data['date_died'].apply(
    lambda x: 0 if x == '9999-99-99' else 1
)


In [4]:
#STEP 4: COUNT DEATHS
death_stats = pd.DataFrame({
    "Count": data['death'].value_counts(),
    "Percentage": data['death'].value_counts(normalize=True) * 100
})

death_stats


,Count,Percentage
death,,
0,530426,93.615271
1,36176,6.384729


In [ ]:
#STEP 5: CLEAN DATA (Considering only Paper Features)
selected_columns = [
    'age',
    'sex',
    'diabetes',
    'obesity',
    'covid_res',
    'death'
]

data_clean = data[selected_columns].copy()

invalid_values = [97, 98, 99]

for col in ['diabetes', 'obesity', 'covid_res']:
    data_clean = data_clean[~data_clean[col].isin(invalid_values)]

data_clean.shape


(564286, 6)

In [6]:
#STEP 6: SPLIT ALIVE & DEATH CASES
death_df = data_clean[data_clean['death'] == 1]
alive_df = data_clean[data_clean['death'] == 0]

num_deaths = len(death_df)

num_deaths, len(alive_df)



(35813, 528473)

In [7]:
#STEP 7: FUNCTION TO CREATE BALANCED DATASET
def create_balanced_dataset(seed):
    alive_sample = alive_df.sample(
        n=num_deaths,
        random_state=seed
    )
    balanced_df = pd.concat([death_df, alive_sample])
    balanced_df = balanced_df.sample(frac=1, random_state=seed)
    return balanced_df


In [8]:
#STEP 8: MODEL TRAINING FUNCTION
def train_and_evaluate(df):
    X = df.drop(columns=['death'])
    y = df['death']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    models = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "SVM": SVC(),
        "KNN": KNeighborsClassifier(),
        "Logistic Regression": LogisticRegression(max_iter=1000)
    }

    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        results[name] = {
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1": f1_score(y_test, y_pred)
        }

    return pd.DataFrame(results).T


In [12]:
#STEP 9: RUN MULTIPLE BALANCED EXPERIMENTS
all_runs = []

seeds = [42, 100, 999, 2021, 7]

for i, seed in enumerate(seeds, start=1):
    balanced_data = create_balanced_dataset(seed)
    run_results = train_and_evaluate(balanced_data)
    run_results['Run'] = f'Run_{i}'
    all_runs.append(run_results)


In [13]:
#STEP 10: COMBINE RESULTS
final_results = pd.concat(all_runs)
final_results


,Accuracy,Precision,Recall,F1,Run
Random Forest,0.788566,0.778496,0.806645,0.792321,Run_1
Decision Tree,0.788147,0.778242,0.805947,0.791852,Run_1
SVM,0.783889,0.769874,0.809856,0.789359,Run_1
KNN,0.772232,0.759723,0.796314,0.777588,Run_1
Logistic Regression,0.786263,0.783023,0.791987,0.787479,Run_1
Random Forest,0.787310,0.767342,0.824654,0.794967,Run_2
Decision Tree,0.786193,0.769029,0.818093,0.792803,Run_2
SVM,0.786751,0.769625,0.818512,0.793316,Run_2
KNN,0.774675,0.762159,0.798548,0.779929,Run_2
Logistic Regression,0.787938,0.778528,0.804830,0.791461,Run_2


In [14]:
#STEP 11: AVERAGE PERFORMANCE ACROSS RUNS
#average_results = final_results.groupby(final_results.index).mean()
#average_results

average_results = final_results.groupby(final_results.index).mean(numeric_only=True)
average_results


,Accuracy,Precision,Recall,F1
Decision Tree,0.787910,0.775046,0.811364,0.792763
KNN,0.770613,0.759985,0.791037,0.775175
Logistic Regression,0.789264,0.782743,0.800810,0.791663
Random Forest,0.788790,0.774401,0.815105,0.794198
SVM,0.784392,0.767845,0.815301,0.790857
